# exp06: Random Oversampling

Menguji pengulangan contoh Low pada training untuk menangani imbalance. Random oversampling mempertahankan nilai fitur contoh asli dan tidak membuat interpolasi baru. Bandingkan XGBoost dan Logistic Regression dengan kontrol tanpa resampling.

Target mengikuti kesulitan tugas: easy sebagai Low, medium sebagai Medium, dan hard sebagai High. Label ini merupakan proxy cognitive load.

### Akses Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Environment

Menyiapkan library pemodelan dengan versi pada `requirements-final.txt`. Fungsi di `shared.experiments` menangani pembentukan fitur, pelatihan, dan evaluasi agar aturan preprocessing konsisten antarpercobaan.

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import hashlib

ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE').resolve()
MODELS_DIR = ROOT / 'models'
manifest_path = MODELS_DIR / 'FINAL_MANIFEST.json'
if not manifest_path.is_file():
    raise RuntimeError('Upload FINAL_MANIFEST.json together with shared and requirements-final.txt')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
for relative, expected in manifest['files'].items():
    if relative.startswith('shared/') or relative == 'requirements-final.txt':
        target = MODELS_DIR / relative
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise RuntimeError(f'Incomplete or outdated upload: {relative}')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(MODELS_DIR / 'requirements-final.txt')])
import importlib.metadata
for module_name, package_name in [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('sklearn', 'scikit-learn'), ('xgboost', 'xgboost'), ('matplotlib', 'matplotlib'), ('joblib', 'joblib'), ('imblearn', 'imbalanced-learn'), ('catboost', 'catboost'), ('threadpoolctl', 'threadpoolctl')]:
    loaded = sys.modules.get(module_name)
    if loaded is not None and getattr(loaded, '__version__', None) != importlib.metadata.version(package_name):
        raise RuntimeError(f'Restart runtime before continuing: {module_name} was loaded before installation')


### Konfigurasi Eksperimen

Menetapkan dataset, seed, direktori hasil, serta mode evaluasi. `full` mengevaluasi seluruh peserta; `smoke` membatasi percobaan untuk pemeriksaan teknis. `RESUME_DIR` menentukan apakah runner membuat run baru atau melanjutkan checkpoint yang cocok.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

sys.path.insert(0, str(MODELS_DIR))
from shared.experiments.runner import Config, run
from shared.experiments.design import registry
from shared.data import load_dataset, describe_data, make_folds

EXPERIMENT = 'exp06'
MODE = 'full'
SEED = 42
RESUME_DIR = None
DATASET_PATH = ROOT / 'Dataset' / 'good_acceptable.csv'
OUTPUT_ROOT = MODELS_DIR / 'outputs'
config = Config(DATASET_PATH, OUTPUT_ROOT, EXPERIMENT, MODE, SEED)

### Dataset dan Target Oversampling

Memuat dataset dan memeriksa distribusi kelas per peserta. Fitur `clean` menghapus `blink_count`, `iris_size_delta_std`, dan `eye_state_other_ratio` karena redundansi atau nilai konstan yang sudah ditemukan. Kedua model memakai fitur clean dan StandardScaler yang sama seperti exp05 dan exp07.

Target jumlah Low adalah 50%, 75%, atau 100% dari jumlah kelas terbesar pada training partition. Target dibulatkan ke atas. Jika Low sudah mencapai target, jumlahnya tetap. Medium dan High tidak ditambah atau dikurangi; validation dan test juga tidak di-resample.

In [ ]:
data = load_dataset(DATASET_PATH)
print(describe_data(data))
display(pd.crosstab(data.participant_id, data.difficulty_level))
folds = make_folds(data)
specs = registry(EXPERIMENT, MODE, SEED)
display(pd.DataFrame([{'model': s['id'], 'features': s['variant'], 'scaler': s['scaler'], 'candidates': len(s['candidates']), 'inner_search': s['tune']} for s in specs]))
print('Outer folds:', len(folds))

### Oversampling di Dalam Training Fold

Setiap peserta menjadi data uji sekali, sementara data peserta lain digunakan untuk training. Identitas peserta hanya dipakai untuk pembagian data, bukan fitur prediksi.

Pada setiap inner training, hapus fitur kosong, fit imputasi median, hapus fitur konstan, lalu fit StandardScaler. Random oversampling mengulang baris Low sampai target tercapai. Latih kandidat rasio dan nilai prediksinya pada inner validation asli.

Pilih rasio berdasarkan macro-F1 gabungan tiga inner validation. Hitung ulang target dari seluruh outer training untuk refit, kemudian prediksi peserta uji. Parameter model tetap; kontrol dilatih tanpa oversampling. Metadata checkpoint menyimpan jumlah kelas sebelum dan sesudah resampling.

In [ ]:
run_dir = run(config, resume_dir=RESUME_DIR)

### Evaluasi Random Oversampling

Macro-F1 dari gabungan prediksi outer test menjadi metrik utama. Accuracy dan balanced accuracy melengkapi precision, recall, dan F1 tiap kelas. Hasil per peserta memperlihatkan apakah performa merata atau hanya baik pada peserta tertentu. Nilai kosong pada metrik kelas berarti peserta uji tidak memiliki contoh kelas tersebut.

Bandingkan varian `balanced` dengan `control` pada model yang sama. Periksa apakah pengulangan Low membantu recall dan F1 Low tanpa menurunkan kelas lain atau precision Low secara berlebihan. Rasio yang terpilih dapat berbeda antar outer fold.

Jumlah baris training yang meningkat tidak berarti bertambahnya peserta atau pengukuran sensor independen. CSV sumber tetap berisi data asli.

In [ ]:
summary = json.loads((run_dir / 'summary.json').read_text())
print(summary)
if summary['status'] != 'complete':
    raise RuntimeError('Incomplete evaluation')
if MODE == 'smoke':
    print('Smoke test only. Change MODE to full and run again for research evaluation.')
display(pd.read_csv(run_dir / 'metrics.csv'))
classes = pd.read_csv(run_dir / 'class_metrics.csv')
display(classes[classes.scope == 'pooled'])
display(pd.read_csv(run_dir / 'participant_metrics.csv'))
display(pd.read_csv(run_dir / 'search_results.csv'))
print('Saved results:', run_dir)

### Confusion Matrix

Baris menunjukkan kelas sebenarnya dan kolom menunjukkan prediksi model. Nilai pada gambar merupakan proporsi per kelas sebenarnya; diagonal menunjukkan prediksi benar. Periksa kelas mana yang sering tertukar, terutama Low dengan Medium atau High.

In [ ]:
for name in pd.read_csv(run_dir / 'metrics.csv').model:
    display(Image(filename=str(run_dir / 'figures' / f'{name}_normalized.png')))